In [ ]:
%reset -f

In [ ]:
import pandas as pd

In [ ]:
def unzip(zipped_list: list[tuple], output_length: int = 1) -> tuple:
    if len(zipped_list) == 0:
        return tuple([] for _ in range(output_length))

    return tuple(map(list, zip(*zipped_list, strict=True)))

print(unzip([(1,2), (3,4), (5,6)]))

In [ ]:
def split_evenely(items: list, classifier_fn):
    partition: dict[int, list] = {}

    # Partition items based on the classifications
    for item in items:
        key = classifier_fn(item)
        if key in partition:
            partition[key].append(item)
        else:
            partition[key] = [item]

    min_count = len(min(partition.values(), key=len))

    # Append the partitioned sublists together up to the min_count
    new_items = []
    for sublist in partition.values():
        new_items += sublist[0:min_count]

    # random.shuffle(new_items)
    return new_items

xs = [-1,1,3,2,1,1,1,2,1,-2,1,2,-3,-3,3,2,2,1,3]
print(split_evenely(xs, lambda x: x*x))

In [ ]:
def split_df_evenly(df: pd.DataFrame, classifier_fn):
    return pd.DataFrame(
        split_evenely((x[1] for x in df.iterrows()), classifier_fn)
    )

In [ ]:
# n = 10
# data = f"balStateData_20000_{n}"

n = 20
# data = f"stateData_2000_{n}_ID0"
# data = f"stateData_2000_{n}_ID1"
data = f"stateData_20000_{n}"

nrows = None

df = pd.read_csv(f"data/{data}.csv", nrows=nrows)
# df = pd.read_csv(f"data/{data}_test.csv", nrows=nrows)

display(df)

In [ ]:
# FIXME: Optimize this to not retrieve whole dataframe subset, just first row of interest
# iter_samples = [
#     df[(df.programIndex == i) & (df.trialIndex == j)].iloc[0]
#     for i in program_indices
#     for j in set(df[df.programIndex == i]["trialIndex"])
# ]
# iter_samples = [
#     (query := df[df.trialIndex == i]).iloc[1 % len(query)]
#     for i in indices
# ]
# iter_samples = [
#     (query := df[df.trialIndex == i]).iloc[2 % len(query)]
#     for i in indices
# ]
# iter_samples = [df[df.trialIndex == i].iloc[-1] for i in indices]

# df[(df.programIndex == 0) & (df.trialIndex == 13)].iloc[0]

iter_index = -1
program_indices = set(df["programIndex"])

iter_samples = []
for i in program_indices:
    trial_indices = set(df[df.programIndex == i]["trialIndex"])

    for j in trial_indices:
        trial_i_rows = df[(df.programIndex == i) & (df.trialIndex == j)]
        if len(trial_i_rows) > iter_index:
            iter_samples.append(trial_i_rows.iloc[iter_index])

# iter_samples = []
# for i in indices:
#     trial_i_rows = df[df.trialIndex == i]
#     if len(trial_i_rows) >= 2:
#         iter_samples.append(trial_i_rows.iloc[0])
#         iter_samples.append(trial_i_rows.iloc[1])

iter_sample_df = pd.DataFrame(iter_samples)
display(iter_sample_df)

In [ ]:
iter_sample_df = iter_sample_df.sample(frac=1).reset_index(drop=True)
display(iter_sample_df)

iter_sample_df.to_csv(f"data/{data}_itr_-1.csv", index=False)

In [ ]:
# bal_df = split_df_evenly(iter_sample_df, lambda row: row["converges"])